In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

In [157]:
df = pd.read_csv("creditcard.csv")
df = df.drop("Time", axis=1)
df = df.drop("Class", axis=1)

In [158]:
df['OutNum'] = 0
old_size = len(df)

for col in df.columns[:-1]:
    Q1 = np.percentile(df[col], 25)
    Q3 = np.percentile(df[col], 75)
    IQR = Q3 - Q1
    lower_bound_iqr = Q1 - 1.5 * IQR
    upper_bound_iqr = Q3 + 1.5 * IQR
    outliers = (df[col] < lower_bound_iqr) | (df[col] > upper_bound_iqr)
    df.loc[outliers, "OutNum"] += 1
    # df[col] = df[col].clip(lower=lower_bound_iqr, upper=upper_bound_iqr)

df = df[df["OutNum"] <= 0.20 * (len(df.columns) - 1)].reset_index(drop=True).iloc[:, :-1] #0.20 is threshold
print((old_size - len(df)) / old_size)

0.05046224285217709


In [159]:
model = IsolationForest(n_estimators=200, contamination='auto', n_jobs=-1, random_state=42)

model.fit(df)
df["noise"] = model.predict(df)

noise = df[df["noise"] == -1]
clean = df[df["noise"] == 1]

print(len(noise) / len(df))
df = clean.drop(columns="noise").reset_index(drop=True)

0.03746926248451569


[0.10454071 0.13296732 0.00228925 ... 0.08385888 0.05199795 0.10445362]
